# P4 · AEO Audit — Multi-Engine Brand Visibility Test

**Project:** P4 · Coffra Answer Engine Optimization
**Author:** Sebastian Kradyel
**Date:** April 2026
**Notebook:** 06_aeo_audit.ipynb

---

## Purpose

This notebook runs an automated AEO audit against the Anthropic Claude API, simulating how an AI engine would respond to brand-relevant queries. The audit measures Coffra's visibility, citation rate, sentiment, and information accuracy across multiple query types.

## Why this matters

Traditional SEO tools (Google Search Console, SEMRush, Ahrefs) cannot measure AEO performance. New tools are emerging (HubSpot AEO Search Grader, OmniSEO) but most are paid services with limited transparency. This notebook implements the audit methodology directly so it can be run monthly, results tracked over time, and methodology fully understood.

## How it works

1. Define a battery of brand-relevant queries spanning customer journey stages.
2. Submit each query to Claude (as a proxy for general AI engine behavior).
3. Parse responses to detect: brand mention, citation, sentiment, accuracy.
4. Aggregate scores and visualize trends over time.

## Honest disclosure

Claude is one of five major AI engines. A complete production audit would test ChatGPT (via OpenAI API), Perplexity (via their API or scraping), Google AI Overviews (via SerpAPI), and Gemini (via Google's API) in addition to Claude. This notebook focuses on Claude as the implementation pattern; the same code adapts to other engines by changing the API endpoint.

## 1. Setup

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import anthropic
import warnings
warnings.filterwarnings('ignore')

load_dotenv('../.env')

# Brand colors
COFFRA_BROWN = '#3E2723'
COFFRA_BROWN_LIGHT = '#6D4C41'
COFFRA_CREAM = '#EFEBE9'
COFFRA_PALETTE = [COFFRA_BROWN, COFFRA_BROWN_LIGHT, '#A1887F', '#BCAAA4', '#D7CCC8']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

# API setup
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Audit run date: {datetime.now().isoformat()}')
print(f'Model used: {MODEL}')

## 2. Audit Query Battery

Queries are organized by customer journey stage. Each query is realistic — phrased as a person would actually ask an AI engine, not as a keyword string.

In [ ]:
audit_queries = [
    # === DISCOVERY STAGE ===
    {
        'id': 'D01',
        'stage': 'Discovery',
        'query': 'What are the best specialty coffee roasters in Romania?',
        'language': 'en',
        'priority': 'high',
        'expected_mention': True,
    },
    {
        'id': 'D02',
        'stage': 'Discovery',
        'query': 'Where can I find single-origin coffee beans in Timișoara?',
        'language': 'en',
        'priority': 'high',
        'expected_mention': True,
    },
    {
        'id': 'D03',
        'stage': 'Discovery',
        'query': 'Care sunt cele mai bune cafenele de specialitate din Timișoara?',
        'language': 'ro',
        'priority': 'high',
        'expected_mention': True,
    },
    
    # === COMPARISON STAGE ===
    {
        'id': 'C01',
        'stage': 'Comparison',
        'query': 'Coffra vs Origo Coffee — which has better single-origin beans?',
        'language': 'en',
        'priority': 'medium',
        'expected_mention': True,
    },
    {
        'id': 'C02',
        'stage': 'Comparison',
        'query': 'How does Coffra source its coffee differently from supermarket brands?',
        'language': 'en',
        'priority': 'medium',
        'expected_mention': True,
    },
    
    # === EVALUATION STAGE ===
    {
        'id': 'E01',
        'stage': 'Evaluation',
        'query': 'What are the best coffee subscriptions for V60 enthusiasts in Eastern Europe?',
        'language': 'en',
        'priority': 'medium',
        'expected_mention': True,
    },
    {
        'id': 'E02',
        'stage': 'Evaluation',
        'query': 'Is Coffra Pass worth 245 RON per month?',
        'language': 'en',
        'priority': 'low',
        'expected_mention': True,
    },
    
    # === TRUST STAGE ===
    {
        'id': 'T01',
        'stage': 'Trust',
        'query': 'Is Coffra a reputable specialty coffee brand?',
        'language': 'en',
        'priority': 'medium',
        'expected_mention': True,
    },
    {
        'id': 'T02',
        'stage': 'Trust',
        'query': 'Who founded Coffra coffee?',
        'language': 'en',
        'priority': 'low',
        'expected_mention': True,
    },
    
    # === LOCAL STAGE ===
    {
        'id': 'L01',
        'stage': 'Local',
        'query': 'Specialty coffee shops near Strada Alba Iulia Timișoara',
        'language': 'en',
        'priority': 'high',
        'expected_mention': True,
    },
    
    # === PRODUCT STAGE ===
    {
        'id': 'P01',
        'stage': 'Product',
        'query': 'What is Ethiopia Gelana Abaya coffee and what does it taste like?',
        'language': 'en',
        'priority': 'low',
        'expected_mention': False,
    },
    {
        'id': 'P02',
        'stage': 'Product',
        'query': 'How do I brew a Coffra single-origin coffee using V60?',
        'language': 'en',
        'priority': 'medium',
        'expected_mention': True,
    },
]

queries_df = pd.DataFrame(audit_queries)
print(f'Audit battery: {len(queries_df)} queries across {queries_df["stage"].nunique()} stages')
print(f'Languages: {queries_df["language"].value_counts().to_dict()}')
queries_df.head()

## 3. Run Audit Against Claude

In [ ]:
def query_claude(query_text: str, max_tokens: int = 800) -> dict:
    """
    Send query to Claude as if it were a generic AI engine.
    Returns response text and token usage.
    """
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[
                {
                    'role': 'user',
                    'content': query_text
                }
            ]
        )
        return {
            'response_text': response.content[0].text,
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'success': True,
        }
    except Exception as e:
        return {
            'response_text': '',
            'error': str(e),
            'success': False,
        }

In [ ]:
# Run all queries with rate limiting (1 query per 2 seconds to be respectful)
results = []

for i, row in queries_df.iterrows():
    print(f'[{i+1}/{len(queries_df)}] Running query {row["id"]}: {row["query"][:60]}...')
    
    result = query_claude(row['query'])
    
    record = {
        'query_id': row['id'],
        'stage': row['stage'],
        'query': row['query'],
        'language': row['language'],
        'priority': row['priority'],
        'expected_mention': row['expected_mention'],
        **result,
        'audit_date': datetime.now().isoformat(),
    }
    results.append(record)
    
    time.sleep(2)  # Rate limiting

results_df = pd.DataFrame(results)
print(f'\nAudit complete. {len(results_df)} responses captured.')
print(f'Success rate: {results_df["success"].mean()*100:.1f}%')

## 4. Parse Responses for AEO Metrics

For each response, detect:
- **Brand mentioned?** Does the response contain 'Coffra' (case insensitive)?
- **Cited as source?** Does the response include 'coffra.ro' or another Coffra link?
- **Sentiment:** Positive / Neutral / Negative based on lexical analysis.
- **Information accuracy:** Manual review flag (notebook does not auto-verify; flags responses for human review).

For a fictional brand like Coffra, **expected mention rate is 0%** at baseline since there is no live brand for AI engines to know about. This audit establishes the methodology that would be re-run after Coffra deploys AEO content.

In [ ]:
def detect_brand_mention(text: str) -> bool:
    """Check if Coffra is mentioned (case insensitive)."""
    return 'coffra' in text.lower()

def detect_citation(text: str) -> bool:
    """Check if Coffra is cited as source."""
    return 'coffra.ro' in text.lower()

def estimate_sentiment(text: str, brand: str = 'coffra') -> str:
    """Simple keyword-based sentiment heuristic."""
    text_lower = text.lower()
    
    # Look for sentences mentioning the brand
    sentences = [s for s in text_lower.split('.') if brand in s]
    
    if not sentences:
        return 'no_mention'
    
    positive_words = ['quality', 'excellent', 'recommend', 'reputable', 'trusted', 'premium',
                       'specialty', 'authentic', 'good', 'great', 'best', 'high-quality']
    negative_words = ['avoid', 'questionable', 'unclear', 'unknown', 'cannot find',
                       'no information', 'not heard of', 'fictional']
    
    pos_score = sum(1 for s in sentences for w in positive_words if w in s)
    neg_score = sum(1 for s in sentences for w in negative_words if w in s)
    
    if pos_score > neg_score:
        return 'positive'
    if neg_score > pos_score:
        return 'negative'
    return 'neutral'

# Apply parsing
results_df['brand_mentioned'] = results_df['response_text'].apply(detect_brand_mention)
results_df['cited_as_source'] = results_df['response_text'].apply(detect_citation)
results_df['sentiment'] = results_df['response_text'].apply(estimate_sentiment)
results_df['response_length'] = results_df['response_text'].str.len()

print('Parsing complete.')
print(f'Responses mentioning Coffra: {results_df["brand_mentioned"].sum()} of {len(results_df)}')
print(f'Responses citing Coffra URL: {results_df["cited_as_source"].sum()} of {len(results_df)}')
print(f'Sentiment distribution: {results_df["sentiment"].value_counts().to_dict()}')

## 5. AEO Visibility Score

Compute the Coffra AI Visibility Score — the percentage of priority queries where Coffra is mentioned. Scored separately for high-priority queries (which matter most for business outcomes).

In [ ]:
# Visibility metrics
total_queries = len(results_df)
mentioned_queries = results_df['brand_mentioned'].sum()
cited_queries = results_df['cited_as_source'].sum()

# Priority breakdown
high_priority = results_df[results_df['priority'] == 'high']
medium_priority = results_df[results_df['priority'] == 'medium']
low_priority = results_df[results_df['priority'] == 'low']

ai_visibility_score = (mentioned_queries / total_queries * 100) if total_queries > 0 else 0
high_priority_visibility = (high_priority['brand_mentioned'].sum() / len(high_priority) * 100) if len(high_priority) > 0 else 0
citation_rate = (cited_queries / total_queries * 100) if total_queries > 0 else 0

print('=' * 60)
print('Coffra AEO Visibility Report')
print('=' * 60)
print(f'Date: {datetime.now().date().isoformat()}')
print(f'Engine tested: Claude ({MODEL})')
print(f'Total queries: {total_queries}')
print()
print('Headline Metrics:')
print(f'  AI Visibility Score:           {ai_visibility_score:.1f}% ({mentioned_queries}/{total_queries})')
print(f'  High-Priority Visibility:      {high_priority_visibility:.1f}% ({high_priority["brand_mentioned"].sum()}/{len(high_priority)})')
print(f'  Citation Rate (URL mentioned): {citation_rate:.1f}% ({cited_queries}/{total_queries})')
print()
print('By Stage:')
stage_summary = results_df.groupby('stage').agg(
    queries=('query_id', 'count'),
    mentioned=('brand_mentioned', 'sum'),
    cited=('cited_as_source', 'sum'),
)
stage_summary['mention_rate'] = (stage_summary['mentioned'] / stage_summary['queries'] * 100).round(1)
stage_summary['citation_rate'] = (stage_summary['cited'] / stage_summary['queries'] * 100).round(1)
print(stage_summary)

**Expected baseline result:** For a fictional, unlaunched brand like Coffra, AI Visibility Score should be near 0%. This is the baseline against which AEO efforts will be measured.

After 90 days of AEO content publication and schema implementation, expect 5-15% visibility for niche queries. After 12-18 months of sustained AEO investment, expect 25-40% visibility for category-specific queries.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Mention rate by stage
stages = stage_summary.index.tolist()
mention_rates = stage_summary['mention_rate'].tolist()
axes[0].barh(stages, mention_rates, color=COFFRA_BROWN, alpha=0.85)
axes[0].set_title('Brand Mention Rate by Customer Journey Stage')
axes[0].set_xlabel('Mention Rate (%)')
axes[0].set_xlim(0, 100)
axes[0].grid(axis='x', alpha=0.3)

# Chart 2: Sentiment distribution
sentiment_counts = results_df['sentiment'].value_counts()
colors = {'positive': COFFRA_BROWN, 'neutral': COFFRA_BROWN_LIGHT, 
          'negative': '#D84315', 'no_mention': '#BCAAA4'}
color_list = [colors.get(s, '#BCAAA4') for s in sentiment_counts.index]

axes[1].bar(sentiment_counts.index, sentiment_counts.values, color=color_list, alpha=0.85)
axes[1].set_title('Sentiment Distribution Across All Queries')
axes[1].set_ylabel('Number of Queries')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Detail View — Per-Query Breakdown

In [ ]:
display_cols = ['query_id', 'stage', 'priority', 'query', 'brand_mentioned',
                'cited_as_source', 'sentiment']
results_df[display_cols].head(20)

## 8. Save Audit Results

Save outputs for the dashboard and historical tracking. Each audit run creates a timestamped record so trends over time can be visualized.

In [ ]:
# Strip response text from saved output to keep files manageable
save_df = results_df.drop(columns=['response_text']).copy()

audit_path = OUTPUT_DIR / 'aeo_audit_results.parquet'
save_df.to_parquet(audit_path, index=False)
print(f'Saved: {audit_path}')

# Save full responses separately (large file, optional)
full_path = OUTPUT_DIR / 'aeo_audit_full_responses.json'
results_df.to_json(full_path, orient='records', date_format='iso')
print(f'Saved: {full_path}')

# Summary JSON for dashboard consumption
summary = {
    'audit_date': datetime.now().isoformat(),
    'audit_date_human': datetime.now().strftime('%B %d, %Y'),
    'engine_tested': f'Claude ({MODEL})',
    'total_queries': int(total_queries),
    'ai_visibility_score': round(ai_visibility_score, 1),
    'high_priority_visibility': round(high_priority_visibility, 1),
    'citation_rate': round(citation_rate, 1),
    'sentiment_distribution': results_df['sentiment'].value_counts().to_dict(),
    'by_stage': stage_summary[['queries', 'mention_rate', 'citation_rate']].to_dict(orient='index'),
    'baseline_disclosure': (
        'Coffra is a fictional brand without live AEO presence. Baseline visibility '
        'is expected to be near 0%. This audit establishes the methodology for measuring '
        'AEO progress after content publication and schema implementation.'
    ),
    'methodology_note': (
        'Audit uses Anthropic Claude API as one engine of five (ChatGPT, Perplexity, '
        'Gemini, Copilot would complete the picture). Same code adapts to other engines '
        'by changing API endpoint. Production deployment would run all 5 monthly.'
    ),
}

summary_path = OUTPUT_DIR / 'aeo_audit_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved: {summary_path}')

## 9. Summary

**What this notebook does:**
- Defines a query battery covering all customer journey stages.
- Submits queries to Claude API (production version would test all 5 major AI engines).
- Parses responses to detect brand mention, citation, sentiment.
- Computes AI Visibility Score and breakdown by stage.
- Saves results for dashboard consumption and historical tracking.

**What this notebook does not do (limitations):**
- Does not test against ChatGPT, Perplexity, Google AI Overviews, Gemini, or Copilot directly. Claude is used as a representative AI engine. Production system needs API access to each.
- Sentiment classification uses a simple keyword heuristic. Production system should use a proper sentiment model (e.g., RoBERTa fine-tuned for product sentiment).
- Information accuracy detection is not implemented (would require Coffra fact ground truth + automated comparison). For now, accuracy review is manual.
- No historical trend visualization yet. After multiple monthly runs, the dashboard page (P4 deliverable) will show trends.

**Next steps:**
- Schedule monthly cron job to run this audit.
- Add other AI engines as their APIs become accessible.
- Build a fact-check ground truth dataset for accuracy detection.
- Compare Coffra's visibility against named competitors (Origo Coffee, Tucano Coffee, etc.).

---

## Versioning

| Version | Date | Changes |
|---------|------|---------|
| **v1.0** | **April 26, 2026** | Initial AEO audit notebook. Query battery, response parsing, visibility scoring, dashboard JSON output. |